In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents.middleware import PIIMiddleware,HumanInTheLoopMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
system_prompt = "You are a helpful assistant that provides information about the user when asked."

In [ ]:
agent=create_agent(model=model,system_prompt=system_prompt,middleware=[PIIMiddleware("email",strategy="redact",apply_to_input=True), PIIMiddleware("credit_card",strategy="hash",apply_to_input=True)])

In [ ]:
response=agent.invoke({'messages': [HumanMessage(content="my email id is kotarigowrish@gmail.com?")]})
print(response)

In [ ]:
config={"configurable": {"thread_id": "hitl_thread_1"}}

Custom PII Patterns

In [ ]:
agent=create_agent(model=model,
                   system_prompt=system_prompt,
                   middleware=[PIIMiddleware("email",strategy="redact",apply_to_input=True), 
                               PIIMiddleware("credit_card",strategy="hash",apply_to_input=True),
                               PIIMiddleware("phone",detector=r"^\+?(\d{1,3})?\s*\(?(\d{1,4})\)?[-.\s]?(\d{1,4})[-.\s]?(\d{1,9})$",strategy="redact",apply_to_input=True),
                               PIIMiddleware("api_key",detector=r"(?i)(api[_-]?key|secret|token)[=:]?[\"']?([a-z0-9]{32,45})[\"']?",strategy="hash",apply_to_input=True),
                               PIIMiddleware("ssn", detector=r"^\d{3}-\d{2}-\d{4}$", strategy="hash", apply_to_input=True)
                               ]
                   )

response=agent.invoke({'messages': [HumanMessage(content="my phone number is +1 (555) 123-4567 and my API key is api_key=1234567890abcdef1234567890abcdef")]})

Human In The Loop

In [ ]:
@tool
def write_to_file(filename: str, content: str) -> str:
    """Writes content to a file."""
    try:
        with open(filename, 'w') as f:
            f.write(content)
    except Exception as e:
        return f"Error writing to file {filename}: {e}"
    return f"Content written to {filename}"
def execute_sql(query: str) -> str:
    """executes a SQL query and returns the results."""
    return f"Executed SQL query: {query}"


In [ ]:
agent=create_agent(model=model,
                   system_prompt=system_prompt,
                   checkpointer=InMemorySaver(),
                   tools=[write_to_file, execute_sql],
                   middleware=[HumanInTheLoopMiddleware(
                       interrupt_on={
                           "write_to_file":True,
                           "execute_sql":{"allowed_decisions":["approve","reject"]},
                       },
                       description_prefix="tool execution pending human approval: "
                    )])

In [ ]:
# Fresh config with new thread to test
config_new = {"configurable": {"thread_id": "hitl_test_fresh3"}}
response_fresh = agent.invoke({'messages': [HumanMessage(content="write hello world to data/test.txt")]}, config=config_new)

if "__interrupt__" in response_fresh:
    interrupt_value = response_fresh["__interrupt__"][0].value
    action_requests = interrupt_value.get('action_requests', [])
    print(f"Number of tool calls: {len(action_requests)}")
    for req in action_requests:
        print(f"  - {req['name']}: {req['args']}")
    
    decisions = [{"type": "approve"} for _ in action_requests]
    
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config_new
    )


In [ ]:

config_multi = {"configurable": {"thread_id": "hitl_multi_tools"}}

response_multi = agent.invoke(
    {'messages': [HumanMessage(content="write hello world to data/test.txt and execute sql query select * from users")]},
    config=config_multi
)

if "__interrupt__" in response_multi:
    interrupt_value = response_multi["__interrupt__"][0].value
    action_requests = interrupt_value.get('action_requests', [])
    print(f"Number of tool calls to approve: {len(action_requests)}\n")
    
    for i, req in enumerate(action_requests):
        print(f"{i+1}. Tool: {req['name']}")
        print(f"   Args: {req['args']}")

    decisions = [
        {"type": "approve"},   # write_to_file - approve
        {"type": "reject"}     # execute_sql - reject
    ]
    
    result = agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config_multi
    )


In [23]:
print(result)

{'messages': [HumanMessage(content='write hello world to data/test.txt and execute sql query select * from users', additional_kwargs={}, response_metadata={}, id='71b86282-a3bd-455a-9bbe-1fcb520084b3'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'execute_sql', 'arguments': '{"query": "select * from users"}'}, '__gemini_function_call_thought_signatures__': {'e8ff1ebd-315a-4de2-bb82-2b9577bbcd37': 'CtACAQw51sev0XWcus3tVOX7ZJmKSNTzGWWy6uM4TyZ3XUyw9LS8ZT7wWfUb7asgqVmrd3QKEYik23b+woVVx5HEtA8RvdCgQucwFebF+UhbrJ1KW2otOvlOzB1Q8EsaSEk7tbK133XxhIH+FsDBL7c7umtbxfCeb9T40AZrhOqcGGmEvBDDNUyv57IRiv3o9kRcrAhGtPNypDlPTX3QT+mOCWyVWyEYFOOZ63HfjBhpReytN9mQ3ZS3gbXBzneTFrC2vimatTejFaPXJB/uIZ0VEf1LzlJiogEAwo+S489dl8Gx6PqxYiKbIU7LlZTSo3o7qL3wJLpWsVoR40ohekySe+20lexuB0Z/n7lGNFedfzf/aNfjihxqsHb5X0SwxaRQm3V8+YDVO3WvGtCdxYmdSkIjjnSxHpWT/FAGwfygKD0TZp0PD4zUCopYqvUXCdlM'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': '